# 04 — Tuning and response shapes

**What you will learn**

- The six inference options every extract/classify route accepts, and where
  they go in the payload.
- What a `threshold` sweep actually trades away, and why the default is not the
  precision dial you might assume.
- How `include_confidence` and `include_spans` **change the response shape**,
  and exactly what changes in your parsing code when you turn them on.
- How to verify that returned character offsets are real, rather than trusting
  the documentation.
- `max_len` and `overlap_policy`, and why you should leave them alone until you
  are genuinely chunking.
- Why a mistyped option key is a `400` rather than a shrug.

**What it assumes you already did**

Notebooks [01](01-getting-started.ipynb), [02](02-extraction-and-classification.ipynb)
and [03](03-structured-and-multitask.ipynb). You should already know the request
shapes for entities, classification and multi-task.

**Roughly how long**

About 25 minutes.

## Setup

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## Where the options go

Every `POST` extract/classify route — single-document and batch alike — accepts
the same optional parameters at the **top level** of the payload, alongside
`text`/`texts` and the task fields. They never go inside `schema_config`; that
object is for the task specification only, which is the separation notebook 03
explained.

| Key | Type | Default | Effect |
|---|---|---|---|
| `threshold` | number in `[0, 1]` | `0.5` | Minimum score for a prediction to survive |
| `include_confidence` | bool | `false` | Add per-prediction scores — **changes response shape** |
| `include_spans` | bool | `false` | Add real character offsets — **changes response shape** |
| `max_len` | positive int | model default | Chunk window for long documents |
| `overlap_policy` | string | model default | How overlapping chunk predictions are reconciled |
| `batch_size` | positive int | model default | Batch routes only; clamped to `MAX_BATCH_SIZE` |

Two of these change the response shape, and that is the part of this notebook
worth reading twice.

## `threshold`, and what a sweep actually costs you

Every prediction carries a score. `threshold` is the floor: predictions scoring
below it are dropped before the response is built. The default is `0.5`.

Here is the thing that surprises people. On clean, unambiguous text this
checkpoint is *extremely* confident — scores sit above `0.99` routinely. Which
means the entire range from `0.2` to `0.9` returns identical results. Raising
your threshold from 0.5 to 0.8 in the hope of "being stricter" does nothing at
all; the model was never producing anything in that band.

The dial does not start biting until you are up near the ceiling. Let us look at
what the model actually believes before sweeping anything, because that is the
step people skip.

In [2]:
AMBIGUOUS = ("The Jaguar was spotted near the old mill road at dusk "
             "by a ranger from Devon.")
AMB_LABELS = ["company", "animal", "person", "location"]

show(post("/extract_entities", {
    "text": AMBIGUOUS,
    "labels": AMB_LABELS,
    "include_confidence": True,
}))

{
  "entities": {
    "company": [],
    "animal": [
      {
        "text": "Jaguar",
        "confidence": 0.9967280626296997
      }
    ],
    "person": [
      {
        "text": "ranger",
        "confidence": 0.97239089012146
      }
    ],
    "location": [
      {
        "text": "old mill road",
        "confidence": 0.9940478801727295
      },
      {
        "text": "Devon",
        "confidence": 0.9691402912139893
      }
    ]
  }
}


In [3]:
print("--- threshold sweep ---")
for th in (0.2, 0.9, 0.98, 0.995, 0.999):
    r = post("/extract_entities", {
        "text": AMBIGUOUS, "labels": AMB_LABELS, "threshold": th,
    })
    kept = {k: v for k, v in r["entities"].items() if v}
    print(f"  threshold={th:<6} kept: {kept}")

--- threshold sweep ---


  threshold=0.2    kept: {'animal': ['Jaguar'], 'person': ['ranger'], 'location': ['old mill road', 'Devon']}


  threshold=0.9    kept: {'animal': ['Jaguar'], 'person': ['ranger'], 'location': ['old mill road', 'Devon']}


  threshold=0.98   kept: {'animal': ['Jaguar'], 'location': ['old mill road']}


  threshold=0.995  kept: {'animal': ['Jaguar']}


  threshold=0.999  kept: {}


### Reading the sweep

Notice the plateau: `0.2` and `0.9` give the same answer. Everything the model
found on this sentence, it found with high confidence. The interesting behavior
is compressed into the last two percent of the range, where predictions start
falling off one at a time in confidence order.

**What a threshold sweep actually trades away is recall, and it trades it for
almost nothing.** This is the crucial point, and it is easy to get backwards.

The reason you would normally raise a threshold is to buy precision — to stop
the model from returning things that are wrong. That works when the model's
errors are *low-confidence* errors, so that a floor separates the wrong answers
from the right ones. Look at what happens here instead: `"old mill road"` as a
location and `"Devon"` as a location are both perfectly correct, and they are
the first things to be cut. By the time you reach `0.999` you have thrown away
everything and kept nothing.

You did not raise precision. You lowered recall and left precision where it was,
because there were no low-confidence errors to remove.

The generalizable rule: **`threshold` only buys you precision if your false
positives actually score lower than your true positives.** Check that with
`include_confidence` on a sample of your own data *before* you tune the dial. If
the wrong answers score as high as the right ones — which is common when a label
is genuinely ambiguous for your domain — no threshold will separate them, and
the fix is a better label, a description (notebook 02), or a different framing
of the task. Not a number.

One more mechanical detail visible above: a label filtered down to nothing stays
in the response as a key with an empty list. It does not disappear. Your parsing
code sees the same key set at any threshold.

## `include_confidence` and `include_spans` change the response shape

This is the section that prevents a specific class of production bug.

Without either flag, an entity is a **bare string**. With either flag, it
becomes an **object**. Your parsing code cannot be agnostic about this — code
that does `for name in result["entities"]["person"]` iterates strings in one
case and dictionaries in the other, and in Python that fails late and
confusingly rather than immediately.

| Flags | Entity element |
|---|---|
| none | `"Tim Cook"` |
| `include_confidence` | `{"text": "Tim Cook", "confidence": 0.99...}` |
| `include_spans` | `{"text": "Tim Cook", "start": 10, "end": 18}` |
| both | `{"text": "Tim Cook", "confidence": ..., "start": ..., "end": ...}` |

Run all four and read them side by side.

In [4]:
SPAN_TEXT = "Apple CEO Tim Cook announced iPhone 15 in Cupertino."
SPAN_LABELS = ["company", "person", "product", "location"]

base = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS})
conf = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                  "include_confidence": True})
spans = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                   "include_spans": True})
both = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                  "include_confidence": True, "include_spans": True})

print("--- bare ---");                 show(base)
print("\n--- include_confidence ---"); show(conf)
print("\n--- include_spans ---");      show(spans)
print("\n--- both ---");               show(both)

--- bare ---
{
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "product": [
      "iPhone 15"
    ],
    "location": [
      "Cupertino"
    ]
  }
}

--- include_confidence ---
{
  "entities": {
    "company": [
      {
        "text": "Apple",
        "confidence": 0.9976344108581543
      }
    ],
    "person": [
      {
        "text": "Tim Cook",
        "confidence": 0.9948838353157043
      }
    ],
    "product": [
      {
        "text": "iPhone 15",
        "confidence": 0.9933070540428162
      }
    ],
    "location": [
      {
        "text": "Cupertino",
        "confidence": 0.997490644454956
      }
    ]
  }
}

--- include_spans ---
{
  "entities": {
    "company": [
      {
        "text": "Apple",
        "start": 0,
        "end": 5
      }
    ],
    "person": [
      {
        "text": "Tim Cook",
        "start": 10,
        "end": 18
      }
    ],
    "product": [
      {
        "text": "iPhone 15",
        "star

### What changes in your parsing code

Concretely, this:

```python
# bare
for name in result["entities"]["person"]:
    print(name.upper())
```

becomes this:

```python
# include_spans and/or include_confidence
for item in result["entities"]["person"]:
    print(item["text"].upper())
```

That is a small change and an easy one to make. The problem is not making it —
it is making it *inconsistently*.

**Decide the shape once per consumer and hold it.** A client that flips these
flags depending on the code path writes two incompatible shapes into the same
dataset, and the failure surfaces months later in whatever reads that dataset,
not in the code that caused it. If you store extraction output, the flag
settings are part of your schema contract, not a per-call convenience.

If you want to be robust to both, normalize at the boundary:

```python
def as_text(item):
    return item if isinstance(item, str) else item["text"]
```

Do that once, at the point where the HTTP response enters your system, not
scattered through the code that consumes it.

Note also that when both flags are on you get **one merged object**, not nested
sub-objects. The keys simply accumulate.

### Verifying that offsets are real

`start` and `end` are genuine character offsets into the text you sent:
`text[start:end]` reproduces the span exactly. That is the property that makes
`include_spans` worth having — with offsets you can highlight in a UI,
deduplicate overlapping mentions, tie a relation argument back to its position,
or attach an extraction to the exact sentence it came from.

Do not take the documentation's word for it. Assert it against the live service.
This is a cheap invariant to add to your own test suite, and it is the kind of
thing that would break silently after a library upgrade.

In [5]:
for label, items in spans["entities"].items():
    for item in items:
        sliced = SPAN_TEXT[item["start"]:item["end"]]
        assert sliced == item["text"], (label, sliced, item)

print("all offsets round-trip against the source text")
print(f"\nexample: text[10:18] == {SPAN_TEXT[10:18]!r}")

all offsets round-trip against the source text

example: text[10:18] == 'Tim Cook'


### Classification changes shape too

Same flag, same principle, different shape: a bare label string becomes a
`{"label", "confidence"}` object.

In [6]:
CLS = {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
}

print("--- bare ---")
show(post("/classify_text", CLS))

print("\n--- include_confidence ---")
show(post("/classify_text", {**CLS, "include_confidence": True}))

--- bare ---


{
  "sentiment": "negative"
}

--- include_confidence ---


{
  "sentiment": {
    "label": "negative",
    "confidence": 0.9999998807907104
  }
}


Classification confidences on clear-cut text are close enough to `1.0` that they
are not very informative on their own. Their real use is as a *routing signal*
in aggregate: extract with confidence, then send anything below some percentile
of your own observed distribution to human review. That is a decision made from
your data, not from a number in a doc.

## `max_len` and `overlap_policy`

These are chunking controls for documents longer than the model's context
window. When a document exceeds the window, the model processes it in
overlapping chunks and then has to reconcile predictions that appear in more
than one chunk — `max_len` sets the window, `overlap_policy` decides the
reconciliation.

Both are forwarded to the `gliner2` library unchanged. The accepted
`overlap_policy` values are the library's, and this API only validates that it
is a non-empty string — so a value the library does not recognize will surface
as an error from deeper in the stack rather than as a clean `400`.

**Leave both unset unless you are actually chunking.** On short documents they
change nothing and only add a way to get it wrong. A smaller `max_len` on text
that already fits is strictly worse: you fragment the context the model uses to
disambiguate, for no benefit. This is the most common self-inflicted quality
regression with these two options.

In [7]:
show(post("/extract_entities", {
    "text": ("Apple CEO Tim Cook announced iPhone 15 in Cupertino. "
             "Satya Nadella leads Microsoft from Redmond."),
    "labels": ["company", "person", "location"],
    "max_len": 384,
    "overlap_policy": "longest",
}))

{
  "entities": {
    "company": [
      "Apple",
      "Microsoft"
    ],
    "person": [
      "Tim Cook",
      "Satya Nadella"
    ],
    "location": [
      "Redmond",
      "Cupertino"
    ]
  }
}


## Bad option values are a `400`, not a shrug

Every one of these options used to be accepted and silently dropped. They are
honoured now, and out-of-range or wrong-typed values are rejected with a message
naming the constraint.

The one that matters most is the **typo case**. Sending `"treshold": 0.5` used
to give you a `200` with default-threshold results — a plausible answer that
quietly ignored your intent. You would tune a parameter, observe no change,
conclude the parameter does not work, and move on with a wrong mental model. Now
it fails immediately and tells you the key was not recognized.

This is the same trade the service makes with `schema_config` keys in notebook
03: a loud failure at the point of the mistake beats a plausible wrong answer
downstream.

In [8]:
bad_payloads = [
    {"threshold": 1.5},            # out of range
    {"threshold": "high"},         # wrong type
    {"include_spans": "yes"},      # not a bool
    {"max_len": 0},                # not positive
    {"treshold": 0.5},             # typo: unknown key
]

for bad in bad_payloads:
    payload = {"text": "Tim Cook.", "labels": ["person"], **bad}
    status, body = post_raw("/extract_entities", payload)
    detail = body.get("detail") if isinstance(body, dict) else body
    print(f"{str(bad):<28} HTTP {status} | {detail}")

{'threshold': 1.5}           HTTP 400 | 'threshold' must be in [0, 1].
{'threshold': 'high'}        HTTP 400 | 'threshold' must be a number in [0, 1].
{'include_spans': 'yes'}     HTTP 400 | 'include_spans' must be a boolean.
{'max_len': 0}               HTTP 400 | 'max_len' must be a positive integer.
{'treshold': 0.5}            HTTP 400 | Unknown payload key(s): ['treshold']. Allowed: ['include_confidence', 'include_spans', 'labels', 'max_len', 'overlap_policy', 'text', 'threshold'].


## Try this yourself

Run a threshold sweep on text from *your* domain with `include_confidence` on,
and answer one question: **do the model's mistakes score lower than its correct
answers?**

The cell below sets that up on a deliberately hard sentence — one where the
model is likely to get something wrong — so you can see what the answer looks
like when it is "no".

In [9]:
HARD = "Amazon shipped Alexa to Washington after Phoenix reviewed the Apple order."
HARD_LABELS = ["company", "person", "location", "product"]

print("--- what the model believes ---")
show(post("/extract_entities", {
    "text": HARD, "labels": HARD_LABELS, "include_confidence": True,
}))

print("\n--- sweep ---")
for th in (0.5, 0.9, 0.98, 0.995, 0.999):
    r = post("/extract_entities", {"text": HARD, "labels": HARD_LABELS, "threshold": th})
    kept = {k: v for k, v in r["entities"].items() if v}
    print(f"  threshold={th:<6} {kept}")

--- what the model believes ---


{
  "entities": {
    "company": [
      {
        "text": "Amazon",
        "confidence": 0.9935232400894165
      },
      {
        "text": "Apple",
        "confidence": 0.9547637701034546
      }
    ],
    "person": [],
    "location": [
      {
        "text": "Washington",
        "confidence": 0.9930591583251953
      },
      {
        "text": "Phoenix",
        "confidence": 0.9276931881904602
      }
    ],
    "product": [
      {
        "text": "Alexa",
        "confidence": 0.9737970232963562
      }
    ]
  }
}

--- sweep ---


  threshold=0.5    {'company': ['Amazon', 'Apple'], 'location': ['Washington', 'Phoenix'], 'product': ['Alexa']}


  threshold=0.9    {'company': ['Amazon', 'Apple'], 'location': ['Washington', 'Phoenix'], 'product': ['Alexa']}


  threshold=0.98   {'company': ['Amazon'], 'location': ['Washington']}


  threshold=0.995  {}


  threshold=0.999  {}


**Discussion.** This sentence is adversarial on purpose: `Amazon` is a company
and a river, `Phoenix` is a city and a plausible person or product name,
`Washington` is a place and a person, `Apple` is a company and a fruit.

Read the confidences, then read the sweep. On this checkpoint the model actually
did fairly well — it declined to return any `person` at all, and its two
lowest-scoring predictions (`Phoenix` at ~0.93, `Apple` at ~0.95) are genuinely
the two most arguable ones. So there *is* some signal in the ordering: the
threshold is not pure noise here.

But watch what it costs to use it. Both of those drop out at `0.98` — and so
does nothing else, because at `0.995` **every** prediction is gone, including
`Amazon` and `Alexa`, which were never in doubt. The usable window between "cuts
the questionable ones" and "cuts everything" is about one and a half percent
wide, and where that window sits depends on the sentence. Tuning a global
threshold into a gap that narrow, on a distribution that moves per document, is
not a robust control.

That is the real lesson, and it is more useful than a rule about whether
threshold "works":

- **Confirm the ordering before you tune.** Run `include_confidence` on a sample
  of your own data and check whether your errors actually sit below your correct
  answers. Sometimes they do, as here. Often, with a genuinely ambiguous label,
  they do not — and then no threshold exists that separates them.
- **Even when the ordering is right, the margin may be too thin to exploit.**
  Look at the gap between your worst true positive and your best false positive.
  If it is a fraction of a percent, you are tuning noise.
- **Prefer fixes that change the distribution over fixes that slice it.** A
  description (notebook 02) that pins down which sense of `company` you mean, a
  structured field with `choices` (notebook 03) that forces the model to commit,
  or simply more surrounding context, all attack the ambiguity itself rather
  than trying to threshold your way around it.

On this checkpoint, on clean text, the default of `0.5` is a reasonable place to
leave it — and reaching for the dial is usually a sign that the label needs work.

## What you learned

- All six inference options go at the **top level** of the payload, never inside
  `schema_config`, and apply to single and batch routes alike.
- This checkpoint is confident: `0.2` through `0.9` typically return identical
  results, and `threshold` only starts filtering in the last couple of percent.
- A threshold sweep trades **recall** away. It buys precision only if your false
  positives score lower than your true positives — verify that with
  `include_confidence` before tuning, because when it is not true the fix is a
  better label or task framing, not a number.
- `include_confidence` and `include_spans` turn bare strings into objects.
  Normalize at the boundary with a single `as_text()` helper, decide the shape
  once per consumer, and treat the flag settings as part of your stored schema.
- Offsets are real: `text[start:end] == item["text"]`. Assert it rather than
  trusting it.
- `max_len` / `overlap_policy` are for genuine chunking only; shrinking the
  window on text that already fits fragments context for no benefit.
- Bad or misspelled option keys are a `400` naming the problem, because a
  silently ignored option is worse than a failed request.

## Next

**[05 — Relation extraction](05-relations.ipynb)** — typed `[head, tail]` pairs,
why they need a boundary checkpoint, and why you must validate them downstream.